# Correction methods of spectral band in EEG synthetic data

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib tk
import numpy as np
import matplotlib.pyplot as plt
from Functions.generate_OU import get_mixed_OU_signals
from Functions.time_frequency import spectrogram
import pywt
import cv2
from sklearn.decomposition import NMF

## 1. Generate synthetic signal

We generate a synthetic signal. In the EEG we want to mimic the presence of $\delta$ + $\alpha$ bands, as it would be for a gabaergic anesthesia and the addition of a $\beta$ band to mimic the __ketamine signature__.

To generate one band we use a __2D Ornstein-Ulhenbeck__ process that can create spindles like patterns with waning and waxing envelopes containing oscillations.

In [86]:
# --- Parameters
T = 20 # desired signal duration (s)
dt = 0.001
fs = 1 / dt

lbda_list = [1, 2, 1]
omega_list = [2*np.pi*1, 2*np.pi*10, 2*np.pi*30]
sigma_list = [3, 2, 2]
factor_list = [1, 1, 1]

# --- Generate EEG data
t, y = get_mixed_OU_signals(T, dt, lbda_list, omega_list, sigma_list, factor_list)

# --- compute spectrogram
f_spectro, t_spectro, spectro = spectrogram(y, fs, nfft_factor=2)
# mask_f = f_spectro >= 20
# f_spectro = f_spectro[mask_f]
# spectro = spectro[mask_f, :]

# --- Display
fig, axes = plt.subplots(3,  constrained_layout = True)
axes[0].plot(t, y)
axes[0].set_title('Simulated EEG signal')
axes[1].pcolormesh(t_spectro, f_spectro, np.log2(spectro + 1e-11), shading = 'nearest', cmap = 'jet')
axes[1].set_title('Spectrogram')
axes[2].plot(f_spectro, np.log2(np.median(spectro, axis = 1)))
axes[2].set_title('PSD')
axes[1].sharex(axes[0])

plt.show()

Select signal as if already segmented within min freq and max freq

In [69]:
M = spectro.copy()
mask_f = f_spectro >= 20
f_M = f_spectro[mask_f]
M = M[mask_f, :]

fig, axis = plt.subplots(1)
axis.pcolormesh(t_spectro, f_M, np.log2(M + 1e-11), shading = 'nearest', cmap = 'jet')

## 2. Methods of correction

__a. Select pixels above psd baseline__

The first step is to detect pixel areas where intensit is high due to ketamine enhancing. For that we make the assumption that the psd should be smooth and have no nbump between left part and right part outside of ketamine frequency range. Using a baseline filter we estimate what should be the clean PSD. Based on this baseline we can define in the freqeuncy range of interest what pixel are ketamine induced by selecting those who are above baseline mulitplied by a factor.

In [70]:
from scipy.sparse import diags
from scipy.sparse.linalg import spsolve

# --- 1. Whittaker-Eilers Baseline ---
def whittaker_baseline(y, lmbda=1e5, p=0.01, n_iter=10):
    n = len(y)
    D = diags([1.0, -2.0, 1.0], [0, 1, 2], shape=(n-2, n), format='csc')
    w = np.ones(n)
    for _ in range(n_iter):
        W = diags([w], [0], shape=(n, n), format='csc')
        A = W + lmbda * D.T @ D
        z = spsolve(A, w * y)
        w = p * (y > z) + (1 - p) * (y <= z)
    return z


Visualize spectrogram mask and PSDs

In [71]:
psd = np.mean(M, axis=1)
baseline = whittaker_baseline(psd, lmbda=1e5, p=0.01)
factor = 2
T_baseline = factor * baseline

In [102]:
# 1. Baseline and threshold calculations
psd_med = np.mean(M, axis=1)
baseline = whittaker_baseline(psd_med, lmbda=1e5, p=0.01)
factor = 2
T_baseline = factor * baseline  # Using your defined factor

# 2. Generate refined hierarchical mask
# Step A: Identify active frequency bins (where average PSD is above threshold)
freq_active = psd_med > 1.5 * baseline  # 1D boolean array of shape (n_frequencies,)

# Step B: Identify candidate pixels above threshold
pixel_mask_candidate = M > (T_baseline[:, np.newaxis]) # M > 3 * (psd_med[:, np.newaxis]) #   # 2D boolean array

# Step C: Refine the mask using broadcasting
# The 1D freq_active array is broadcasted horizontally across all time steps
mask = pixel_mask_candidate & freq_active[:, np.newaxis]

# 3. Separate M into remaining and masked parts preserving 2D shape using NaN
M_remaining = np.where(~mask, M, np.nan)
M_masked = np.where(mask, M, np.nan)

# 4. Compute PSDs (mean over time axis, ignoring NaNs)
remaining_psd = np.nanmean(M_remaining, axis=1)
masked_psd = np.nanmean(M_masked, axis=1)

C:\Users\holcman\AppData\Local\Temp\ipykernel_30036\206005872.py:24: RuntimeWarning: Mean of empty slice
  masked_psd = np.nanmean(M_masked, axis=1)


In [103]:
# Plotting (4 subplots now)
fig, axes = plt.subplots(4, figsize=(10, 16), sharex=False)

# Subplot 0: Original PSD, Baseline, and Threshold
axes[0].semilogy(f_M, psd_med, label='Original PSD (Mean)')
axes[0].semilogy(f_M, baseline, label='Baseline')
axes[0].semilogy(f_M, T_baseline, label='Threshold (3x Baseline)')
axes[0].legend(loc='upper right')
axes[0].set_ylabel('Power')
axes[0].set_title('PSD & Thresholds')

# Subplot 1: Original Spectrogram
im1 = axes[1].pcolormesh(t_spectro, f_M, np.log2(M + 1e-11), shading='nearest', cmap='jet')
fig.colorbar(im1, ax=axes[1], label='log2 Power')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Original Spectrogram')

# Subplot 2: Mask
im2 = axes[2].pcolormesh(t_spectro, f_M, mask, shading='nearest', cmap='jet')
fig.colorbar(im2, ax=axes[2], label='Mask State')
axes[2].set_ylabel('Frequency')
axes[2].set_title('Boolean Mask (True/1 = Above Threshold)')

# Subplot 3: PSD Comparison (Remaining vs Masked)
axes[3].semilogy(f_M, remaining_psd, label='Remaining PSD (Below Threshold)', color='green')
axes[3].semilogy(f_M, masked_psd, label='Masked PSD (Above Threshold)', color='red')
axes[3].semilogy(f_M, baseline, label='Baseline')
axes[3].semilogy(f_M, T_baseline, label='Threshold (3x Baseline)')
axes[3].legend(loc='upper right')
axes[3].set_xlabel('Frequency')
axes[3].set_ylabel('Power')
axes[3].set_title('PSD: Remaining vs. Masked Spectrograms')

plt.tight_layout()
plt.show()


__b. Methods of corrections__

we have two constraints:
- we want to preserve intensity ordering of pixel across different frequencies and time.
- a pixel intensity cannot go lower than its baseline threshold for a given frequency.

In [121]:
import numpy as np
import matplotlib.pyplot as plt
import warnings

# ==========================================
# 1. Baseline and threshold calculations
# ==========================================
psd_med = np.mean(M, axis=1)
baseline = whittaker_baseline(psd_med, lmbda=1e5, p=0.01)
T_baseline = 1 * baseline
threshold_matrix = np.tile(T_baseline[:, np.newaxis], (1, M.shape[1]))

# ==========================================
# 2. Generate mask
# ==========================================
mask = M > threshold_matrix

# ==========================================
# 3. Monotonic Floor Projection Compression
# ==========================================
M_transformed = M.copy()

if np.any(mask):
    # Step A: Extract original values and corresponding thresholds inside the mask
    X = M[mask]
    Y = threshold_matrix[mask]
    
    # Step B: Sort by the original pixel values to ensure we process in order of intensity
    sort_idx = np.argsort(X)
    unsort_idx = np.argsort(sort_idx) # Used to return values to their original spatial positions
    
    X_sorted = X[sort_idx]
    Y_sorted = Y[sort_idx]
    
    # Step C: Compute the cumulative maximum of the thresholds to build the monotonic floor
    F = np.maximum.accumulate(Y_sorted)
    
    # Step D: Apply compression using beta
    # beta = 0.10 means we compress 90% of the headroom, pulling peaks tightly to the baseline floor
    beta = 0.1
    X_sorted_compressed = (1 - beta) * F + beta * X_sorted
    
    # Step E: Unsort the compressed values back to their original pixel positions
    M_transformed[mask] = X_sorted_compressed[unsort_idx]

# ==========================================
# 4. Separate transformed M for PSD analysis (keeping 2D shape)
# ==========================================
M_remaining = np.where(~mask, M_transformed, np.nan)
M_masked = np.where(mask, M_transformed, np.nan)

# Compute max of remaining spectrogram safely
if np.isnan(M_remaining).all():
    max_remaining_value = np.nan
else:
    max_remaining_value = np.nanmax(M_remaining)

# Compute PSDs (mean over time axis, ignoring NaNs)
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=RuntimeWarning)
    remaining_psd = np.nanmean(M_remaining, axis=1)
    masked_psd = np.nanmean(M_masked, axis=1)

# Compute global transformed PSD
psd_transformed = np.mean(M_transformed, axis=1)

# ==========================================
# 5. Plotting (4 subplots)
# ==========================================
fig, axes = plt.subplots(4, figsize=(10, 18), sharex=False)

# Subplot 0: Original PSD vs New Transformed PSD
axes[0].semilogy(f_M, psd_med, label='Original PSD (Mean)', alpha=0.5)
axes[0].semilogy(f_M, psd_transformed, label='Transformed PSD (Mean)', color='purple', linewidth=2)
axes[0].semilogy(f_M, baseline, label='Baseline')
axes[0].semilogy(f_M, T_baseline, label='Threshold (2x Baseline)')
if not np.isnan(max_remaining_value):
    axes[0].axhline(max_remaining_value, color='r', linestyle=':', label=f'Max Remaining ({max_remaining_value:.2e})')
axes[0].legend(loc='upper right')
axes[0].set_ylabel('Power')
axes[0].set_title('PSD & Thresholds (With Monotonic Floor Projection)')

# Subplot 1: Transformed Spectrogram
im1 = axes[1].pcolormesh(t_spectro, f_M, np.log2(M_transformed + 1e-11), shading='nearest', cmap='jet')
fig.colorbar(im1, ax=axes[1], label='log2 Power')
axes[1].set_ylabel('Frequency')
axes[1].set_title(f'Transformed Spectrogram (Compressed with $\\beta$={beta:.2f})')

# Subplot 2: Mask
im2 = axes[2].pcolormesh(t_spectro, f_M, mask, shading='nearest', cmap='jet')
fig.colorbar(im2, ax=axes[2], label='Mask State')
axes[2].set_ylabel('Frequency')
axes[2].set_title('Boolean Mask')

# Subplot 3: PSD Comparison
axes[3].semilogy(f_M, remaining_psd, label='Remaining PSD (Below Threshold)', color='green')
axes[3].semilogy(f_M, masked_psd, label='Compressed Masked PSD', color='red')
axes[3].semilogy(f_M, baseline, label='Baseline')
axes[3].semilogy(f_M, T_baseline, label='Threshold (2x Baseline)')
axes[3].legend(loc='upper right')
axes[3].set_xlabel('Frequency')
axes[3].set_ylabel('Power')
axes[3].set_title('PSD: Remaining vs. Compressed Masked Spectrograms')

plt.tight_layout()
plt.show()

Plot before after spectrogram

In [99]:
fig, axes = plt.subplots(2, sharex = True, constrained_layout =True)
im0 = axes[0].pcolormesh(t_spectro, f_M, np.log2(M + 1e-11), shading='nearest', cmap='jet', vmin = -20, vmax = -5)
fig.colorbar(im0, ax=axes[0], label='log2 Power')
axes[0].set_ylabel('Frequency')
axes[0].set_title(f'original Spectrogram')

im1 = axes[1].pcolormesh(t_spectro, f_M, np.log2(M_transformed + 1e-11), shading='nearest', cmap='jet', vmin = -20, vmax = -5)
fig.colorbar(im1, ax=axes[1], label='log2 Power')
axes[1].set_ylabel('Frequency')
axes[1].set_title(f'Transformed Spectrogram (Compressed with $\\beta$={beta:.2f})')

plt.show()

In [116]:
import numpy as np
import matplotlib.pyplot as plt
import warnings

# ==========================================
# 1. Baseline and threshold calculations
# ==========================================
psd_med = np.mean(M, axis=1)
baseline = whittaker_baseline(psd_med, lmbda=1e5, p=0.01)
T_baseline = 0.5 * baseline
threshold_matrix = np.tile(T_baseline[:, np.newaxis], (1, M.shape[1]))

# ==========================================
# 2. Generate mask
# ==========================================
mask = M > threshold_matrix

threshold_matrix = np.tile(0.5*baseline[:, np.newaxis], (1, M.shape[1]))


# ==========================================
# 3. Normalized Headroom Compression (Fractional-Safe)
# ==========================================
M_transformed = M.copy()

if np.any(mask):
    # Extract original values and corresponding thresholds inside the mask
    X = M[mask]
    Y = threshold_matrix[mask]
    
    # Sort by the original pixel values to ensure we process in order of intensity
    sort_idx = np.argsort(X)
    unsort_idx = np.argsort(sort_idx)  # Used to map sorted values back to original positions
    
    X_sorted = X[sort_idx]
    Y_sorted = Y[sort_idx]
    
    # Compute the cumulative maximum of the thresholds to build the monotonic floor
    F = np.maximum.accumulate(Y_sorted)
    
    # Compute the original headroom
    headroom = X_sorted - F
    H_max = np.max(headroom)
    
    if H_max > 0:
        # Define compression settings
        S = 0.10     # Reduce the maximum peak's headroom by 90%
        gamma = 1.5  # Exponent >= 1.0 safely compresses fractional normalized values
        
        # Apply the normalized power-law compression
        normalized_headroom = headroom / H_max
        compressed_headroom = H_max * S * (normalized_headroom ** gamma)
        
        # Reconstruct the compressed values
        X_sorted_compressed = F + compressed_headroom
    else:
        X_sorted_compressed = X_sorted
    
    # Unsort the compressed values back to their original pixel positions
    M_transformed[mask] = X_sorted_compressed[unsort_idx]

# ==========================================
# 4. Separate transformed M for PSD analysis (keeping 2D shape)
# ==========================================
M_remaining = np.where(~mask, M_transformed, np.nan)
M_masked = np.where(mask, M_transformed, np.nan)

# Compute max of remaining spectrogram safely
if np.isnan(M_remaining).all():
    max_remaining_value = np.nan
else:
    max_remaining_value = np.nanmax(M_remaining)

# Compute PSDs (mean over time axis, ignoring NaNs)
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=RuntimeWarning)
    remaining_psd = np.nanmean(M_remaining, axis=1)
    masked_psd = np.nanmean(M_masked, axis=1)

# Compute global transformed PSD
psd_transformed = np.mean(M_transformed, axis=1)

# ==========================================
# 5. Plotting (4 subplots)
# ==========================================
fig, axes = plt.subplots(4, figsize=(10, 18), sharex=False)

# Subplot 0: Original PSD vs New Transformed PSD
axes[0].semilogy(f_M, psd_med, label='Original PSD (Mean)', alpha=0.5)
axes[0].semilogy(f_M, psd_transformed, label='Transformed PSD (Mean)', color='purple', linewidth=2)
axes[0].semilogy(f_M, baseline, label='Baseline')
axes[0].semilogy(f_M, T_baseline, label='Threshold (2x Baseline)')
if not np.isnan(max_remaining_value):
    axes[0].axhline(max_remaining_value, color='r', linestyle=':', label=f'Max Remaining ({max_remaining_value:.2e})')
axes[0].legend(loc='upper right')
axes[0].set_ylabel('Power')
axes[0].set_title('PSD & Thresholds (With Normalized Headroom Compression)')

# Subplot 1: Transformed Spectrogram
im1 = axes[1].pcolormesh(t_spectro, f_M, np.log2(M_transformed + 1e-11), shading='nearest', cmap='jet')
fig.colorbar(im1, ax=axes[1], label='log2 Power')
axes[1].set_ylabel('Frequency')
axes[1].set_title(f'Transformed Spectrogram (Peak Scale $S$={S:.2f}, Exponent $\\gamma$={gamma:.2f})')

# Subplot 2: Mask
im2 = axes[2].pcolormesh(t_spectro, f_M, mask, shading='nearest', cmap='jet')
fig.colorbar(im2, ax=axes[2], label='Mask State')
axes[2].set_ylabel('Frequency')
axes[2].set_title('Boolean Mask')

# Subplot 3: PSD Comparison
axes[3].semilogy(f_M, remaining_psd, label='Remaining PSD (Below Threshold)', color='green')
axes[3].semilogy(f_M, masked_psd, label='Compressed Masked PSD', color='red')
axes[3].semilogy(f_M, baseline, label='Baseline')
axes[3].semilogy(f_M, T_baseline, label='Threshold (2x Baseline)')
axes[3].legend(loc='upper right')
axes[3].set_xlabel('Frequency')
axes[3].set_ylabel('Power')
axes[3].set_title('PSD: Remaining vs. Compressed Masked Spectrograms')

plt.tight_layout()
plt.show()

In [101]:
fig, axes = plt.subplots(2, sharex = True, constrained_layout =True)
im0 = axes[0].pcolormesh(t_spectro, f_M, np.log2(M + 1e-11), shading='nearest', cmap='jet', vmin = -20, vmax = -5)
fig.colorbar(im0, ax=axes[0], label='log2 Power')
axes[0].set_ylabel('Frequency')
axes[0].set_title(f'original Spectrogram')

im1 = axes[1].pcolormesh(t_spectro, f_M, np.log2(M_transformed + 1e-11), shading='nearest', cmap='jet', vmin = -20, vmax = -5)
fig.colorbar(im1, ax=axes[1], label='log2 Power')
axes[1].set_ylabel('Frequency')
axes[1].set_title(f'Transformed Spectrogram (Compressed with $\\beta$={beta:.2f})')

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage.morphology import reconstruction


spectrogram = M
initial_mask = mask

# ==============================================================================
# 2. PERFORM SAFE REGION GROWING TO 10TH PERCENTILE
# ==============================================================================

# Calculate the 10th percentile threshold of the spectrogram
threshold = np.percentile(spectrogram, 10)

# The allowed growth zone: mask expansion is physically constrained to stay 
# within areas strictly higher than the 10th percentile.
allowed_zone = spectrogram > threshold

# CRITICAL FIX: Ensure seed values are strictly <= mask values.
# If any starting pixel in the initial_mask falls below the 10th percentile, 
# we logically force it to False so the reconstruction algorithm does not crash.
corrected_seed = initial_mask.astype(bool) & allowed_zone

# Perform morphological reconstruction by dilation
# It expands outward from the seed, but is strictly stopped at the 'allowed_zone' boundaries.
extended_mask = reconstruction(seed=corrected_seed, 
                               mask=allowed_zone, 
                               method='dilation').astype(bool)

# ==============================================================================
# 3. COMPUTE MASKED MEAN (With safe warning/nan handling)
# ==============================================================================

# Apply the extended mask to the spectrogram
M_masked = np.where(extended_mask, spectrogram, np.nan)

# Compute the mean across the time axis (axis=1)
# Using np.errstate suppresses the RuntimeWarning for rows that contain only NaNs.
with np.errstate(all='ignore'):
    masked_psd = np.nanmean(M_masked, axis=1)
    # Safely convert empty slice NaNs to 0.0
    masked_psd = np.nan_to_num(masked_psd, nan=0.0)

# ==============================================================================
# 4. PLOT THE RESULTS
# ==============================================================================
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Plot 1: Original Spectrogram
im0 = axes[0].imshow(spectrogram, aspect='auto', origin='lower', cmap='viridis')
axes[0].set_title("Original Spectrogram")
fig.colorbar(im0, ax=axes[0], label="Power")

# Plot 2: Initial starting mask (red border overlay)
axes[1].imshow(spectrogram, aspect='auto', origin='lower', cmap='viridis', alpha=0.7)
axes[1].contour(initial_mask, colors='red', levels=[0.5], linewidths=2)
axes[1].set_title("Initial Mask (Red)")

# Plot 3: Resulting extended mask
axes[2].imshow(spectrogram, aspect='auto', origin='lower', cmap='viridis', alpha=0.7)
axes[2].contour(extended_mask, colors='magenta', levels=[0.5], linewidths=2)
axes[2].set_title("Extended Mask (Magenta)\nStopped at < 10th %ile")

for ax in axes:
    ax.set_xlabel("Time Bins")
    ax.set_ylabel("Frequency Bins")

plt.tight_layout()
plt.show()